In [1]:
import pandas as pd
import geopandas as geopd
import numpy as np
import os
import pathlib
from tqdm.notebook import tqdm
from utils.timeseries_utils import read_daily_timeseries_csv

# This script processes and combines timeseries data for catchments.
# It reads timeseries data from CSV files, combines relevant attributes, and saves the combined data for each catchment.

In [2]:
# Order of attributes to be included in the final timeseries
order_list = ['discharge_vol', 'discharge_spec', 'precipitation', 'pet', 'pe_era5_land', 'pet_fmi', 'snow_evaporation', 'swe', 'swe_cci3-1', 'snow_depth', 'temperature_gmin', 'temperature_min', 'temperature_mean', 'temperature_max', 'humidity_rel', 'radiation_global']

calculate_pet = False

# Paths to the catchment and timeseries data
catchments_path = "/path/to/CAMELS_FI_catchments.gpkg"
src_root = "/path/to/timeseries_by_attribute"
dst_root = "/path/to/timeseries"
catchments = geopd.read_file(catchments_path, layer='catchments')

# Get the list of files in the source directory
root_path = pathlib.Path(src_root)
files = [item for item in root_path.iterdir() if item.is_file()]

In [ ]:
# Combining snow evaporation, FMI potential evaporation and ERA5 potential evaporation based on if snow depth > 0

In [3]:
gauges = list(catchments['gauge_id'])

if calculate_pet:
    for file in files:
        attribute = file.stem

        if attribute == 'pet_fmi':
            pet_fmi = read_daily_timeseries_csv(file)
            pet_fmi = pet_fmi[gauges]
            
        if attribute == 'pet_era5_land':
            pet_era5 = read_daily_timeseries_csv(file)
            pet_era5 = pet_era5[gauges]
            
        if attribute == 'snow_evaporation':
            snow_e = read_daily_timeseries_csv(file)
            snow_e = snow_e[gauges]
            
        if attribute == 'snow_depth':
            snow_depth = read_daily_timeseries_csv(file)
            # snow depth has some extra gauges that have been removed
            snow_depth = snow_depth[gauges]
            
    assert pet_era5.columns.equals(snow_e.columns), "Columns don't match"
    assert pet_era5.columns.equals(snow_depth.columns), "Columns don't match"
    assert pet_era5.index.equals(snow_e.index), "Indices don't match"
    assert pet_era5.index.equals(snow_depth.index), "Indices don't match"

    # Snow evaporation is used for snowy days
    pet = pd.DataFrame(np.where(snow_depth > 0, snow_e, np.nan), index=pet_era5.index, columns=pet_era5.columns)
    # Filling non-snowy observations with FMI pet,
    pet = pet.fillna(pet_fmi)
    # then the gaps with era5-land pet
    pet = pet.fillna(pet_era5)

    # Limiting the valid range to 1981
    pet = pet.loc["1981":"2023"]
    
    pet_path = os.path.join(src_root, 'pet.csv')
    pet_path = pathlib.Path(pet_path)
    pet.to_csv(pet_path)
    if pet_path not in files:
        files.append(pet_path)
    

In [5]:
for gauge in tqdm(gauges):
    columns = []
    for file in files:
        attribute = file.stem
        meteo = read_daily_timeseries_csv(file)

        column = meteo.loc["1961":"2023", [gauge]]
        column = column.rename({gauge: attribute}, axis=1)
        columns.append(column)
        
     # Combine the columns into a single DataFrame    
    output = pd.concat(columns, axis=1)
    output = output[order_list]

    # Save the combined timeseries data for the gauge
    dst_path = os.path.join(dst_root, f"CAMELS_FI_hydromet_timeseries_{gauge}_19610101-20231231.csv")
    output.to_csv(dst_path)

  0%|          | 0/320 [00:00<?, ?it/s]